# Projeto de Machine Learning — Fase Inicial (CRISP-DM)
## Previsão de Cancelamento de Clientes (*Churn*) em uma Operadora de Telecomunicações

**Disciplina:** Machine Learning  
**Atividade 1:** Entendimento do Negócio e Entendimento dos Dados  
**Equipe:** *(preencher: nome dos 6 integrantes)*

---

**Sumário**

1. Entendimento do Negócio
2. Entendimento dos Dados
   - 2.1 Fonte dos dados
   - 2.2 Descrição geral da base
   - 2.3 Dicionário de dados
   - 2.4 Análise Exploratória de Dados (EDA)
   - 2.5 Visualizações e interpretações
3. Síntese e próximos passos

In [ ]:
# Bibliotecas utilizadas
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100

SEED = 42
np.random.seed(SEED)

# 1. Entendimento do Negócio

## 1.1 Contextualização do problema

**Cenário (domínio de aplicação).** O projeto se situa no setor de **telecomunicações**, mais especificamente em uma operadora que oferece serviços de **telefonia fixa e internet** a clientes residenciais. Esse é um mercado maduro, de alta concorrência e com produtos muito parecidos entre si, em que o cliente pode trocar de operadora com facilidade.

**Motivação prática.** Em empresas de assinatura, o cancelamento de clientes (*churn*) corrói diretamente a receita recorrente. Além de perder a receita futura, a empresa perde o investimento já feito para conquistar aquele cliente. Na literatura de marketing, é amplamente aceito que **reter um cliente costuma ser mais barato do que adquirir um novo**. Por isso, saber *quem tem maior risco de sair* antes de a saída acontecer permite agir de forma preventiva (oferta personalizada, contato do time de retenção, ajuste de plano).

**Motivação científica.** A previsão de churn é um problema clássico de **classificação binária com classes desbalanceadas**, com variáveis de naturezas mistas (demográficas, contratuais, de uso de serviços e financeiras). Ele permite exercitar toda a cadeia do CRISP-DM e discutir questões metodológicas relevantes, como **desbalanceamento de classes**, **escolha de métricas** além da acurácia e **vazamento de dados (*data leakage*)**.

**Relevância.** Mesmo pequenas reduções na taxa de churn têm impacto financeiro relevante quando multiplicadas pela base de clientes e pelo valor de vida de cada um. Um modelo que priorize os clientes de maior risco permite concentrar o orçamento limitado de retenção onde ele rende mais, em vez de aplicar descontos indiscriminadamente.

## 1.2 Definição do problema

**Problema a ser resolvido:** dado o perfil de um cliente (dados demográficos, serviços contratados e informações de conta), **prever se ele irá cancelar o serviço** no período analisado.

**Tipo de problema:** **Classificação binária**.

- *Por que classificação?* A variável de interesse é categórica com duas classes (o cliente saiu ou permaneceu).
- *Por que não regressão?* Não queremos estimar um valor numérico contínuo, e sim uma decisão entre duas categorias. (Embora modelos de classificação possam devolver uma probabilidade de churn, usada para ordenar clientes por risco, o alvo continua sendo categórico.)
- *Por que não agrupamento?* Já existe um rótulo conhecido, que é justamente o que queremos prever. O agrupamento poderia ser usado de forma complementar (segmentação de perfis), mas não responde à pergunta central.

## 1.3 Objetivo da solução

**Saída esperada do algoritmo:**

- **Classe prevista:** `1` (cliente cancela / *churn*) ou `0` (cliente permanece);
- **Probabilidade de churn** de cada cliente, permitindo montar uma lista priorizada para a equipe de retenção.

**Objetivos específicos:**

1. Identificar os fatores mais associados ao cancelamento (por exemplo, tipo de contrato, tempo de relacionamento, forma de pagamento);
2. Construir, nas próximas fases, um modelo capaz de antecipar o churn com desempenho superior a um modelo ingênuo (*baseline*).

## 1.4 Tipo de aprendizado

**Aprendizado de máquina supervisionado.**

**Justificativa:** a base de dados contém o **rótulo verdadeiro** de cada cliente (a coluna `Churn Value`/`Churn Label`, que indica se ele saiu ou ficou). Como temos exemplos históricos com a resposta correta, o algoritmo pode aprender a relação entre as características dos clientes e o desfecho. Uma abordagem não supervisionada não faria uso dessa informação, que é o ponto mais valioso da base.

## 1.5 Critérios de sucesso

**Impacto esperado:** reduzir a taxa de cancelamento por meio de ações de retenção direcionadas aos clientes de maior risco.

**Critérios técnicos (a serem verificados na fase de Modelagem/Avaliação):**

| Critério | Justificativa |
|---|---|
| Superar o *baseline* (prever sempre a classe majoritária) | Se a acurácia do modelo não superar a de um modelo que "chuta" sempre "permanece", ele não agrega valor. |
| **Recall** da classe *churn* | Perder um cliente que iria sair (falso negativo) costuma custar mais do que oferecer um benefício a quem não sairia (falso positivo). |
| **F1-score** e **Precision** | Evitam gastar orçamento de retenção com muitos falsos alarmes. |
| **ROC-AUC / PR-AUC** | Medem a capacidade de ordenar clientes por risco, independentemente do limiar escolhido. |
| Interpretabilidade | A área de negócio precisa entender *por que* um cliente é considerado de risco para agir sobre isso. |

> **Observação metodológica:** como a base é desbalanceada, a **acurácia isolada não é uma boa métrica**. Isso será demonstrado na seção de EDA ao analisarmos a distribuição da variável-alvo.

# 2. Entendimento dos Dados

## 2.1 Fonte dos dados

- **Base:** *Telco Customer Churn — IBM Dataset*
- **Origem:** Kaggle, a partir de dados de exemplo publicados pela IBM (*IBM Cognos Analytics sample data — Telco Churn*)
- **Link:** https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset
- **Natureza dos dados:** dados **fictícios**, que simulam uma operadora de telefonia fixa e internet com clientes na Califórnia (EUA). Por serem sintéticos/de exemplo, não há dados pessoais reais, mas é preciso lembrar que os resultados **não** representam o comportamento de uma operadora real.

**Como reproduzir:** o notebook procura o arquivo `Telco_customer_churn.xlsx` na mesma pasta. Caso ele não exista, tenta baixá-lo automaticamente com a biblioteca `kagglehub` (requer internet). Se nenhuma das opções funcionar, baixe manualmente pelo link acima e coloque o arquivo na pasta do notebook.

In [ ]:
def localizar_arquivo():
    """Procura o arquivo da base localmente; se não achar, tenta baixar via kagglehub."""
    nomes = ["Telco_customer_churn.xlsx", "Telco_customer_churn.csv"]
    for n in nomes:
        if os.path.exists(n):
            return n

    try:
        import kagglehub
        pasta = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
        for n in nomes:
            achados = glob.glob(os.path.join(pasta, "**", n), recursive=True)
            if achados:
                return achados[0]
    except Exception as e:
        print("Download automático indisponível:", type(e).__name__)

    raise FileNotFoundError(
        "Arquivo não encontrado. Baixe 'Telco_customer_churn.xlsx' em "
        "https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset "
        "e coloque na mesma pasta deste notebook."
    )

caminho = localizar_arquivo()
print("Arquivo utilizado:", caminho)

df = pd.read_excel(caminho) if caminho.endswith(".xlsx") else pd.read_csv(caminho)
df.columns = df.columns.str.strip()
df.head()

## 2.2 Descrição geral da base

In [ ]:
n_linhas, n_colunas = df.shape
print(f"Número de instâncias (linhas): {n_linhas:,}")
print(f"Número de atributos (colunas): {n_colunas}")
print()

print("Tipos de dados (dtype) por coluna:")
print(df.dtypes.value_counts().to_string())

In [ ]:
# Classificação semântica das variáveis (base do dicionário de dados e da EDA)
alvo = "Churn Value"

col_identificacao  = ["CustomerID"]
col_geograficas    = ["Country", "State", "City", "Zip Code", "Lat Long", "Latitude", "Longitude"]
col_numericas      = ["Tenure Months", "Monthly Charges", "Total Charges"]
col_categoricas    = ["Gender", "Senior Citizen", "Partner", "Dependents", "Phone Service",
                      "Multiple Lines", "Internet Service", "Online Security", "Online Backup",
                      "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies",
                      "Contract", "Paperless Billing", "Payment Method"]
col_pos_churn      = ["Churn Label", "Churn Score", "CLTV", "Churn Reason"]   # ver seção 2.4.7

# Conferência: todas as colunas esperadas existem?
esperadas = (col_identificacao + col_geograficas + col_numericas + col_categoricas
             + col_pos_churn + [alvo])
faltando = [c for c in esperadas if c not in df.columns]
extras   = [c for c in df.columns if c not in esperadas]
print("Colunas esperadas ausentes:", faltando if faltando else "nenhuma")
print("Colunas não classificadas :", extras if extras else "nenhuma")

resumo_tipos = pd.DataFrame({
    "Grupo": ["Identificação", "Geográficas", "Numéricas", "Categóricas",
              "Resultado/pós-churn", "Alvo"],
    "Qtde": [len(col_identificacao), len(col_geograficas), len(col_numericas),
             len(col_categoricas), len(col_pos_churn), 1],
})
resumo_tipos

**Resumo dos tipos de dado:** a base mistura variáveis **numéricas** (tempo de contrato e valores cobrados), **categóricas** (serviços contratados, tipo de contrato, forma de pagamento, dados demográficos), **geográficas** (cidade, CEP, latitude/longitude) e um **identificador** de cliente. Não há variáveis de texto livre para análise, exceto `Churn Reason`, que é um motivo textual preenchido apenas para quem cancelou.

## 2.3 Dicionário de dados

| Variável | Tipo | Descrição |
|---|---|---|
| `CustomerID` | Identificador | Código único do cliente (não tem valor preditivo). |
| `Count` | Constante | Valor 1 em todas as linhas (campo técnico da fonte). |
| `Country`, `State` | Categórica | País e estado do cliente (valores constantes: EUA e Califórnia). |
| `City`, `Zip Code` | Categórica | Cidade e CEP do cliente. |
| `Lat Long`, `Latitude`, `Longitude` | Geográfica | Coordenadas da residência do cliente. |
| `Gender` | Categórica | Gênero do cliente (Male/Female). |
| `Senior Citizen` | Categórica | Indica se o cliente é idoso (Yes/No). |
| `Partner` | Categórica | Indica se o cliente tem cônjuge/parceiro. |
| `Dependents` | Categórica | Indica se o cliente tem dependentes. |
| `Tenure Months` | Numérica | Meses de relacionamento do cliente com a empresa. |
| `Phone Service` | Categórica | Possui serviço de telefonia fixa. |
| `Multiple Lines` | Categórica | Possui múltiplas linhas telefônicas. |
| `Internet Service` | Categórica | Tipo de internet contratada (DSL, Fiber optic ou No). |
| `Online Security` | Categórica | Possui serviço de segurança online. |
| `Online Backup` | Categórica | Possui serviço de backup online. |
| `Device Protection` | Categórica | Possui proteção de dispositivo. |
| `Tech Support` | Categórica | Possui suporte técnico. |
| `Streaming TV` | Categórica | Possui serviço de TV por streaming. |
| `Streaming Movies` | Categórica | Possui serviço de filmes por streaming. |
| `Contract` | Categórica | Tipo de contrato (Month-to-month, One year, Two year). |
| `Paperless Billing` | Categórica | Fatura digital (sem papel). |
| `Payment Method` | Categórica | Forma de pagamento (cheque eletrônico, cheque enviado por correio, transferência bancária automática, cartão de crédito automático). |
| `Monthly Charges` | Numérica | Valor cobrado mensalmente (US$). |
| `Total Charges` | Numérica | Valor total cobrado ao longo do relacionamento (US$). |
| `Churn Label` | Alvo (texto) | "Yes" se o cliente saiu; "No" se permaneceu. |
| **`Churn Value`** | **Alvo (0/1)** | **Versão numérica do `Churn Label`: 1 = saiu, 0 = permaneceu. É a variável que se pretende prever.** |
| `Churn Score` | Pós-churn | Pontuação de 0 a 100 gerada por um modelo preditivo da IBM (SPSS Modeler). Quanto maior, maior a probabilidade de churn. |
| `CLTV` | Pós-churn | *Customer Lifetime Value*: valor de vida do cliente estimado pela IBM. |
| `Churn Reason` | Pós-churn | Motivo do cancelamento, preenchido apenas para quem saiu. |

> **Atenção:** `Churn Label`, `Churn Score`, `CLTV` e `Churn Reason` **não devem ser usadas como variáveis preditoras**. Elas são resultado do desfecho ou de outro modelo que já o estima, e usá-las geraria **vazamento de dados**. Isso é discutido na seção 2.4.7.

## 2.4 Análise Exploratória de Dados (EDA)

### 2.4.1 Estrutura e qualidade geral

In [ ]:
df.info()

In [ ]:
# Duplicidades
print("Linhas duplicadas (linha inteira):", df.duplicated().sum())
print("CustomerID duplicados            :", df["CustomerID"].duplicated().sum())

# Colunas constantes (sem informação para o modelo)
constantes = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print("Colunas constantes:", constantes)

### 2.4.2 Valores ausentes

Valores ausentes podem aparecer de duas formas: como `NaN` (explícitos) ou como **texto vazio/espaços** (disfarçados). Vamos verificar as duas.

In [ ]:
# 1) Ausentes explícitos (NaN)
ausentes_nan = df.isna().sum()

# 2) Ausentes disfarçados (strings vazias ou só com espaços) em colunas de texto
ausentes_branco = {}
for c in df.select_dtypes(include=["object", "string"]).columns:
    ausentes_branco[c] = df[c].astype(str).str.strip().eq("").sum()
ausentes_branco = pd.Series(ausentes_branco).reindex(df.columns, fill_value=0)

tabela_aus = pd.DataFrame({
    "NaN": ausentes_nan,
    "Vazio/espaço": ausentes_branco,
})
tabela_aus["Total"] = tabela_aus["NaN"] + tabela_aus["Vazio/espaço"]
tabela_aus["% do total"] = tabela_aus["Total"] / len(df) * 100
tabela_aus = tabela_aus[tabela_aus["Total"] > 0].sort_values("Total", ascending=False)
tabela_aus

In [ ]:
# 'Total Charges' costuma vir como texto por conter espaços em branco: investigar
print("dtype de Total Charges:", df["Total Charges"].dtype)

df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
print("Ausentes em Total Charges após conversão numérica:", df["Total Charges"].isna().sum())

# Quem são esses clientes?
sem_total = df[df["Total Charges"].isna()]
print()
print("Tenure Months dos clientes sem Total Charges:")
print(sem_total["Tenure Months"].value_counts().to_string() if len(sem_total) else "nenhum caso")

**Interpretação — valores ausentes.**

- Há dois tipos de "ausência" nesta base, e **ambos são estruturais**, não falhas aleatórias de coleta:
  1. `Total Charges` possui poucos registros em branco. Eles correspondem a clientes com **`Tenure Months = 0`**, ou seja, clientes recém-chegados que ainda **não receberam nenhuma fatura**. O valor correto, portanto, é **0** (e não um dado "perdido"). Na modelagem, a decisão mais coerente será imputar `0` em vez de usar média ou remover as linhas.
  2. `Churn Reason` só é preenchido para clientes que **cancelaram**; para quem permaneceu o campo é vazio por definição. Isso reforça que essa coluna só existe *depois* do desfecho.
- Não há necessidade de descartar colunas por excesso de ausentes, mas o tratamento deve ser **consciente do significado** de cada ausência.

### 2.4.3 Estatísticas descritivas

In [ ]:
# Numéricas
df[col_numericas + ["Churn Score", "CLTV"]].describe().T

In [ ]:
# Categóricas
df[col_categoricas].describe().T

In [ ]:
# Frequência de cada categoria (visão compacta)
for c in col_categoricas:
    freq = df[c].value_counts(dropna=False)
    print(f"{c}: " + " | ".join(f"{k}: {v} ({v/len(df)*100:.1f}%)" for k, v in freq.items()))

**Interpretação — estatísticas descritivas.**

- `Tenure Months` varia de 0 a cerca de 72 meses (6 anos), o que mostra uma base com clientes novos e antigos.
- `Monthly Charges` e `Total Charges` têm escalas muito diferentes. `Total Charges` é, em essência, **aproximadamente `Tenure Months × Monthly Charges`**, o que antecipa **forte correlação** entre essas três variáveis (verificada na seção 2.5.6) e indica que modelos sensíveis à escala (regressão logística, KNN, redes neurais) exigirão **padronização**.
- A maior parte das variáveis categóricas é **binária** (Yes/No); as demais têm 3 a 4 níveis, o que facilita a codificação (*one-hot*) sem gerar explosão de dimensionalidade.
- Muitas variáveis de serviço (`Online Security`, `Tech Support` etc.) têm a categoria **"No internet service"**, que é redundante com `Internet Service = No`. Isso gera **multicolinearidade** entre variáveis que precisará ser considerada.

### 2.4.4 Distribuição da variável-alvo

In [ ]:
contagem = df[alvo].value_counts().sort_index()
percentual = df[alvo].value_counts(normalize=True).sort_index() * 100

tabela_alvo = pd.DataFrame({
    "Classe": ["0 - Permaneceu", "1 - Cancelou (churn)"],
    "Clientes": contagem.values,
    "%": percentual.values,
})
display(tabela_alvo) if "display" in globals() else print(tabela_alvo)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=alvo, data=df, ax=ax, palette=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["Permaneceu (0)", "Cancelou (1)"])
ax.set_xlabel("")
ax.set_ylabel("Número de clientes")
ax.set_title("Distribuição da variável-alvo (Churn)")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}\n({p.get_height()/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, contagem.max() * 1.15)
plt.tight_layout()
plt.show()

baseline_acc = percentual.max() / 100
print(f"Acurácia de um modelo ingênuo (prever sempre a classe majoritária): {baseline_acc:.1%}")

**Interpretação — variável-alvo.**

- A base é **desbalanceada**: aproximadamente **73% dos clientes permanecem** e **27% cancelam**. Esse desbalanceamento é moderado, mas relevante.
- Um modelo "preguiçoso" que **sempre prevê "permanece"** já teria **~73% de acurácia**, sem identificar nenhum cliente em risco. Portanto, **acurácia não é uma métrica adequada** para este problema, o que justifica os critérios definidos na seção 1.5 (Recall, F1, ROC-AUC/PR-AUC).
- Na fase de modelagem, será necessário usar **divisão estratificada** (treino/teste) e avaliar técnicas como ponderação de classes (`class_weight`) ou reamostragem.

### 2.4.5 Distribuição das variáveis numéricas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, c in enumerate(col_numericas):
    sns.histplot(df[c].dropna(), bins=30, kde=True, ax=axes[0, i], color="#4C72B0")
    axes[0, i].set_title(f"Histograma — {c}")
    axes[0, i].set_xlabel("")
    sns.boxplot(x=df[c], ax=axes[1, i], color="#8EC1DA")
    axes[1, i].set_title(f"Boxplot — {c}")
    axes[1, i].set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Assimetria e presença de outliers (regra do IQR)
linhas = []
for c in col_numericas:
    s = df[c].dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    outliers = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    linhas.append({"Variável": c, "Média": s.mean(), "Mediana": s.median(),
                   "Assimetria (skew)": s.skew(), "Outliers (IQR)": outliers})
pd.DataFrame(linhas)

**Interpretação — variáveis numéricas.**

- **`Tenure Months`** apresenta uma distribuição **em forma de "U"**: há uma grande concentração de clientes **muito novos** (primeiros meses) e outra de clientes **muito antigos** (~72 meses), com pouca gente no meio. Isso sugere dois perfis bem distintos de clientes (os recém-chegados e os fidelizados).
- **`Monthly Charges`** é **bimodal**: há um grupo com cobrança muito baixa (provavelmente apenas telefonia ou internet básica) e outro de cobrança mais alta (pacotes com vários serviços e fibra óptica).
- **`Total Charges`** é **assimétrica à direita** (cauda longa): muitos clientes com valores acumulados baixos e poucos com valores altos. Isso decorre de ser um valor acumulado no tempo.
- Pela regra do IQR, **não há outliers relevantes**: os valores extremos são legítimos (clientes antigos com muitos serviços) e **não devem ser removidos**.

### 2.4.6 Distribuição das variáveis categóricas

In [ ]:
n = len(col_categoricas)
colunas_fig = 4
linhas_fig = int(np.ceil(n / colunas_fig))
fig, axes = plt.subplots(linhas_fig, colunas_fig, figsize=(20, 4.2 * linhas_fig))
axes = axes.flatten()

for ax, c in zip(axes, col_categoricas):
    ordem = df[c].value_counts().index
    sns.countplot(y=c, data=df, order=ordem, ax=ax, color="#4C72B0")
    ax.set_title(c)
    ax.set_xlabel("Clientes")
    ax.set_ylabel("")
for ax in axes[n:]:
    ax.axis("off")
plt.suptitle("Distribuição das variáveis categóricas", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

**Interpretação — variáveis categóricas.**

- A maioria dos clientes tem **contrato mensal (*Month-to-month*)**. Esse é um ponto de atenção: contratos sem fidelidade tornam o cancelamento muito mais fácil.
- Quanto à internet, as categorias **Fiber optic** e **DSL** são as principais, e uma parcela menor não tem internet.
- `Gender` é praticamente **equilibrada** entre homens e mulheres; já `Senior Citizen` é uma minoria da base.
- A forma de pagamento mais comum é o **cheque eletrônico** (*Electronic check*), com as demais formas bem distribuídas.
- Serviços adicionais (`Online Security`, `Tech Support`, `Device Protection` etc.) têm adesão **minoritária**. Isso indica espaço comercial para *upsell*, e, como veremos, esses serviços parecem estar ligados à retenção.

### 2.4.7 Variáveis de vazamento de dados (*data leakage*)

Antes de analisar a relação das variáveis com o churn, é preciso identificar quais delas **não poderiam ser usadas para prever** o cancelamento em um cenário real.

In [ ]:
# Churn Label é só a versão textual do alvo?
print("Concordância entre Churn Label e Churn Value:")
display(pd.crosstab(df["Churn Label"], df[alvo])) if "display" in globals() else print(pd.crosstab(df["Churn Label"], df[alvo]))

# Churn Reason só existe para quem cancelou?
print("\nChurn Reason preenchido, por classe do alvo:")
preench = df.groupby(alvo)["Churn Reason"].apply(lambda s: s.notna().mean() * 100)
print(preench.round(1).to_string())

# Churn Score e CLTV vs alvo
print("\nMédia de Churn Score e CLTV por classe do alvo:")
print(df.groupby(alvo)[["Churn Score", "CLTV"]].mean().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(x=alvo, y="Churn Score", data=df, ax=axes[0], palette=["#4C72B0", "#C44E52"])
axes[0].set_xticklabels(["Permaneceu", "Cancelou"]); axes[0].set_xlabel("")
axes[0].set_title("Churn Score por classe")
sns.boxplot(x=alvo, y="CLTV", data=df, ax=axes[1], palette=["#4C72B0", "#C44E52"])
axes[1].set_xticklabels(["Permaneceu", "Cancelou"]); axes[1].set_xlabel("")
axes[1].set_title("CLTV por classe")
plt.tight_layout()
plt.show()

**Interpretação — vazamento de dados.**

- **`Churn Label`** é apenas o próprio alvo escrito em texto (concordância de 100% com `Churn Value`). Usá-la como preditora seria "dar a resposta ao modelo".
- **`Churn Reason`** só é preenchida para quem **já cancelou**, portanto só existe *depois* do evento que queremos prever.
- **`Churn Score`** é a saída de **outro modelo preditivo** da IBM, e os boxplots mostram que ele separa quase perfeitamente as duas classes. Usá-lo faria nosso modelo apenas "copiar" o modelo anterior, com desempenho artificialmente alto e sem valor prático.
- **`CLTV`** é calculado pela IBM com base em informações que podem incorporar o desfecho, e tem comportamento diferente entre as classes.

**Decisão do grupo:** essas quatro colunas serão **excluídas do conjunto de atributos preditores** na fase de modelagem. Elas continuam úteis **para a análise exploratória** (por exemplo, entender os motivos de cancelamento), mas não para o treino. Também serão excluídas `CustomerID` (identificador), `Count`, `Country` e `State` (constantes).

## 2.5 Visualizações e interpretações

### 2.5.1 Taxa de churn por variável categórica

Para cada categoria, calculamos o **percentual de clientes que cancelaram**. A linha tracejada indica a taxa média geral. Barras acima dela indicam grupos de **risco acima da média**.

In [ ]:
taxa_geral = df[alvo].mean() * 100

fig, axes = plt.subplots(linhas_fig, colunas_fig, figsize=(20, 4.2 * linhas_fig))
axes = axes.flatten()

for ax, c in zip(axes, col_categoricas):
    taxa = (df.groupby(c)[alvo].mean() * 100).sort_values(ascending=False)
    sns.barplot(x=taxa.values, y=taxa.index.astype(str), ax=ax, color="#C44E52")
    ax.axvline(taxa_geral, color="black", linestyle="--", linewidth=1)
    ax.set_title(c)
    ax.set_xlabel("Taxa de churn (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, max(60, taxa.max() * 1.15))
    for p, v in zip(ax.patches, taxa.values):
        ax.annotate(f"{v:.1f}%", (v, p.get_y() + p.get_height()/2), va="center",
                    ha="left", fontsize=9, xytext=(3, 0), textcoords="offset points")
for ax in axes[n:]:
    ax.axis("off")
plt.suptitle(f"Taxa de churn por categoria (linha tracejada = média geral: {taxa_geral:.1f}%)",
             y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

**Interpretação — taxa de churn por categoria.**

Os padrões mais marcantes costumam ser:

- **Tipo de contrato (`Contract`)** é o fator mais discriminante: clientes em **contrato mensal** cancelam em taxa **muito superior** à média, enquanto contratos de **1 e 2 anos** têm taxas baixíssimas. A fidelização contratual funciona como forte barreira de saída.
- **Internet por fibra óptica (`Internet Service = Fiber optic`)** concentra churn bem acima do DSL e de quem não tem internet. Pode indicar insatisfação com preço/qualidade da fibra ou maior exposição à concorrência, hipótese que deve ser investigada com o negócio.
- **Serviços de proteção e suporte** (`Online Security`, `Tech Support`, `Online Backup`, `Device Protection`): clientes que **não contratam** esses serviços cancelam bem mais do que quem contrata. Suporte e segurança parecem aumentar o vínculo e a percepção de valor.
- **Forma de pagamento**: quem paga por **cheque eletrônico** apresenta churn claramente superior às formas automáticas (cartão e débito automático), que reduzem o atrito de renovação e se associam a clientes mais engajados.
- **Perfil do cliente**: ter **parceiro** e **dependentes** está associado a menor churn (vínculos familiares tendem a estabilizar a assinatura), e **idosos** cancelam mais do que a média. **`Gender`** e **`Phone Service`** praticamente **não discriminam** o churn (taxas muito próximas entre as categorias), sendo candidatas a pouca relevância preditiva.
- **Fatura digital (`Paperless Billing`)** está associada a churn um pouco maior, provavelmente refletindo um perfil de cliente mais "digital" e com maior facilidade para trocar de operadora.

> *Os percentuais exatos aparecem nos gráficos acima; as conclusões devem ser conferidas com os números gerados na execução.*

### 2.5.2 Variáveis numéricas × churn

In [ ]:
rotulos = {0: "Permaneceu", 1: "Cancelou"}
df["_status"] = df[alvo].map(rotulos)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, c in enumerate(col_numericas):
    sns.histplot(data=df, x=c, hue="_status", bins=30, kde=True, stat="density",
                 common_norm=False, ax=axes[0, i], palette=["#4C72B0", "#C44E52"])
    axes[0, i].set_title(f"{c} — distribuição por classe")
    sns.boxplot(data=df, x="_status", y=c, ax=axes[1, i], palette=["#4C72B0", "#C44E52"])
    axes[1, i].set_title(f"{c} — boxplot por classe")
    axes[1, i].set_xlabel("")
plt.tight_layout()
plt.show()

df[[alvo] + col_numericas].groupby(alvo).agg(["mean", "median"]).round(1).T

**Interpretação — numéricas × churn.**

- **`Tenure Months`**: quem cancela tem **tempo de casa bem menor** (mediana de poucos meses, contra mais de 2 anos para quem fica). O risco de churn é maior **nos primeiros meses de relacionamento**, o que sugere priorizar ações de *onboarding* e retenção no início da jornada.
- **`Monthly Charges`**: clientes que cancelam pagam, em média, **mensalidades mais altas**. Combinado ao resultado da fibra óptica, isso sugere que **preço percebido** é um motivador de saída.
- **`Total Charges`**: quem cancela tem valor acumulado menor, mas isso é em grande parte **efeito do menor tempo de casa** (variável derivada de `Tenure × Monthly Charges`) e não uma causa independente.

### 2.5.3 Taxa de churn por faixa de tempo de contrato

In [ ]:
bins = [-1, 6, 12, 24, 48, 72]
rot  = ["0–6 meses", "7–12", "13–24", "25–48", "49–72"]
df["_faixa_tenure"] = pd.cut(df["Tenure Months"], bins=bins, labels=rot)
taxa_faixa = df.groupby("_faixa_tenure", observed=True)[alvo].agg(["mean", "count"])
taxa_faixa["mean"] *= 100

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=taxa_faixa.index.astype(str), y=taxa_faixa["mean"], ax=ax, color="#C44E52")
ax.axhline(taxa_geral, color="black", linestyle="--", linewidth=1, label=f"Média geral ({taxa_geral:.1f}%)")
for p, (_, r) in zip(ax.patches, taxa_faixa.iterrows()):
    ax.annotate(f"{r['mean']:.1f}%\n(n={int(r['count'])})", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, taxa_faixa["mean"].max() * 1.25)
ax.set_xlabel("Tempo de relacionamento")
ax.set_ylabel("Taxa de churn (%)")
ax.set_title("Taxa de churn por faixa de tempo de contrato")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretação.** A taxa de churn **decresce de forma consistente com o tempo de relacionamento**: é mais alta na faixa inicial (0–6 meses) e cai nas faixas mais longas. Isso confirma que **o início da jornada é o período crítico** e que a relação entre `Tenure Months` e churn é monotônica e forte, portanto uma variável com alto poder preditivo esperado.

### 2.5.4 Combinação de contrato e tempo de relacionamento

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="Contract", y="Tenure Months", hue="_status", ax=ax,
            palette=["#4C72B0", "#C44E52"], order=["Month-to-month", "One year", "Two year"])
ax.set_title("Tempo de relacionamento por tipo de contrato e status")
ax.set_xlabel("")
ax.legend(title="")
plt.tight_layout()
plt.show()

**Interpretação.** Mesmo **dentro de cada tipo de contrato**, quem cancela tem menor tempo de casa. Contudo, o efeito do contrato é visível: nos contratos longos, os poucos cancelamentos ocorrem em geral mais cedo e há **pouquíssimas saídas**. Isso mostra que `Contract` e `Tenure Months` carregam informação **complementar e parcialmente sobreposta**, o que deve ser considerado na modelagem (interações e colinearidade).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="Internet Service", y="Monthly Charges", hue="_status", ax=ax,
            palette=["#4C72B0", "#C44E52"])
ax.set_title("Mensalidade por tipo de internet e status")
ax.set_xlabel("")
ax.legend(title="")
plt.tight_layout()
plt.show()

**Interpretação.** A **fibra óptica** é o plano de maior mensalidade, e é nesse grupo que os clientes que cancelam pagam valores ainda maiores. O risco de churn parece estar ligado à **combinação de preço alto com ausência de serviços de suporte**, e não apenas ao tipo de tecnologia.

### 2.5.5 Motivos de cancelamento (somente quem saiu)

In [ ]:
motivos = df.loc[df[alvo] == 1, "Churn Reason"].value_counts().head(12)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=motivos.values, y=motivos.index, ax=ax, color="#C44E52")
ax.set_xlabel("Clientes que cancelaram")
ax.set_ylabel("")
ax.set_title("Principais motivos de cancelamento (Top 12)")
for p, v in zip(ax.patches, motivos.values):
    ax.annotate(str(v), (v, p.get_y() + p.get_height()/2), va="center", ha="left",
                xytext=(3, 0), textcoords="offset points", fontsize=9)
plt.tight_layout()
plt.show()

**Interpretação.** Os motivos mais frequentes se concentram em **concorrência** (ofertas, velocidade e dados de outras operadoras), **qualidade do atendimento/atitude dos funcionários** e **preço**. Isso indica que boa parte do churn é **evitável**: está ligada a fatores sobre os quais a empresa tem controle (suporte, preço e oferta). Reforça também o valor de um modelo preditivo que permita agir antes da decisão do cliente. Lembrando que esta variável é **só para análise**, e não para treino (seção 2.4.7).

### 2.5.6 Correlação entre variáveis numéricas

In [ ]:
col_corr = col_numericas + [alvo, "Latitude", "Longitude"]
corr = df[col_corr].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=.5, ax=ax)
ax.set_title("Matriz de correlação (Pearson) — variáveis numéricas e alvo")
plt.tight_layout()
plt.show()

print("Correlação de cada variável com o alvo (Churn Value):")
print(corr[alvo].drop(alvo).sort_values().round(3).to_string())

**Interpretação — correlações.**

- **`Tenure Months` × `Total Charges`** apresentam correlação **muito alta e positiva**, como já antecipado: o valor total é função do tempo de casa. Há **redundância** entre elas.
- **`Total Charges` × `Monthly Charges`** também têm correlação positiva alta, pelo mesmo motivo.
- Em relação ao alvo, **`Tenure Months`** tem correlação **negativa moderada** (quanto maior o tempo de casa, menor o churn), e `Monthly Charges` tem correlação **positiva fraca**. As correlações lineares com o alvo são modestas, o que **não significa pouca relevância**: muitas relações são não lineares ou categóricas (como o tipo de contrato).
- **Latitude e Longitude** têm correlação praticamente nula com o churn, e a localização geográfica, isoladamente, não parece explicar o cancelamento.

### 2.5.7 Associação entre variáveis categóricas e o churn (V de Cramér)

A correlação de Pearson não se aplica a variáveis categóricas. Usamos o **V de Cramér** (0 = nenhuma associação; 1 = associação perfeita), derivado do teste qui-quadrado.

In [ ]:
def cramers_v(x, y):
    tabela = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabela, correction=False)[0]
    n = tabela.values.sum()
    r, k = tabela.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

assoc = pd.Series({c: cramers_v(df[c], df[alvo]) for c in col_categoricas}).sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(x=assoc.values, y=assoc.index, ax=ax, color="#4C72B0")
ax.set_xlabel("V de Cramér com o churn")
ax.set_title("Força de associação das variáveis categóricas com o churn")
for p, v in zip(ax.patches, assoc.values):
    ax.annotate(f"{v:.2f}", (v, p.get_y() + p.get_height()/2), va="center", ha="left",
                xytext=(3, 0), textcoords="offset points", fontsize=9)
plt.tight_layout()
plt.show()

**Interpretação.** O ranking confirma o que os gráficos de taxa já indicavam: **`Contract`** lidera com folga, seguida por variáveis como **`Online Security`, `Tech Support`, `Internet Service`, `Payment Method`** e `Dependents`. Já **`Gender`** e **`Phone Service`** aparecem no fim do ranking, com associação próxima de zero, sendo candidatas à **exclusão ou baixa prioridade** na seleção de atributos.

### 2.5.8 Distribuição geográfica dos clientes

In [ ]:
amostra = df.sample(min(len(df), 4000), random_state=SEED)

fig, ax = plt.subplots(figsize=(7, 7))
sns.scatterplot(data=amostra, x="Longitude", y="Latitude", hue="_status", s=15, alpha=.6,
                palette={"Permaneceu": "#4C72B0", "Cancelou": "#C44E52"}, ax=ax)
ax.set_title("Localização dos clientes (amostra) por status")
ax.legend(title="")
plt.tight_layout()
plt.show()

top_cidades = (df.groupby("City")[alvo].agg(["count", "mean"])
                 .query("count >= 30").sort_values("mean", ascending=False).head(10))
top_cidades["mean"] = (top_cidades["mean"] * 100).round(1)
top_cidades.columns = ["Clientes", "Taxa de churn (%)"]
top_cidades

**Interpretação.** Os clientes se distribuem por todo o estado, com **concentração nas grandes regiões metropolitanas**, e **não há um padrão espacial claro** separando churn de não churn. Algumas cidades têm taxa de churn mais alta, mas, com poucos clientes por cidade, parte dessa variação é **ruído amostral**. Por isso, `City` e `Zip Code` (alta cardinalidade) provavelmente agregarão pouco e podem aumentar o risco de *overfitting*.

# 3. Síntese e próximos passos

## 3.1 Principais descobertas

| # | Descoberta | Implicação para a modelagem |
|---|---|---|
| 1 | Base desbalanceada (~27% de churn) | Usar divisão estratificada; métricas Recall, F1, PR-AUC/ROC-AUC; avaliar `class_weight`/reamostragem. |
| 2 | `Total Charges` tem valores em branco que correspondem a clientes com `Tenure = 0` | Converter para numérico e imputar **0**. |
| 3 | `Churn Label`, `Churn Score`, `CLTV` e `Churn Reason` causam vazamento | **Excluir** das variáveis preditoras. |
| 4 | Contrato mensal, fibra óptica, pagamento por cheque eletrônico, ausência de suporte/segurança e baixo tempo de casa estão associados a maior churn | Variáveis prioritárias; testar interações (ex.: contrato × tenure). |
| 5 | `Tenure`, `Monthly Charges` e `Total Charges` são fortemente correlacionadas | Atenção à multicolinearidade; considerar remover `Total Charges` ou usar modelos robustos a ela. |
| 6 | `Gender`, `Phone Service` e localização têm pouca associação com o churn | Candidatas a descarte na seleção de atributos. |
| 7 | Categoria "No internet service" é redundante com `Internet Service = No` | Padronizar/agrupar em "No" antes da codificação. |

## 3.2 Próximas fases (CRISP-DM)

1. **Preparação dos Dados:** limpeza, tratamento de `Total Charges`, codificação (*one-hot*), padronização, divisão treino/teste estratificada, tratamento do desbalanceamento.
2. **Modelagem:** *baseline*, Regressão Logística, Árvore de Decisão, Random Forest e Gradient Boosting, com validação cruzada.
3. **Avaliação:** Recall, Precision, F1, ROC-AUC, PR-AUC, matriz de confusão e análise de custo de erro.
4. **Implantação:** discussão de como a lista de clientes de maior risco seria usada pela área de retenção.

## 3.3 Limitações

- Os dados são **fictícios/de exemplo**, e as conclusões não se generalizam para uma operadora real.
- Trata-se de um **recorte transversal** (um trimestre): não há histórico de comportamento ao longo do tempo (uso, reclamações, chamados), que costuma ser muito informativo para churn.
- As associações observadas são **correlacionais**, e não necessariamente causais.

In [ ]:
# Limpeza de colunas auxiliares criadas apenas para os gráficos
df = df.drop(columns=["_status", "_faixa_tenure"], errors="ignore")
print("Notebook concluído. Dimensão final do DataFrame:", df.shape)